# Importação de Bibliotecas

In [ ]:
import os
import gc
from datetime import datetime
from math import trunc
import time

import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LeakyReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier
import joblib


In [ ]:
# Checagem de disponibilidade da GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ {len(gpus)} GPU(s) disponível(is): {[gpu.name for gpu in gpus]}")
else:
    print("⚠️ Nenhuma GPU disponível. Usando CPU.")

✅ 1 GPU(s) disponível(is): ['/physical_device:GPU:0']


# Cria Matriz de Confusão

In [ ]:
def cria_CM_Macro(nome_grupo, pasta_cm, modelo_nome, y_true_all_folds, y_pred_all_folds):
    y_true_all = np.concatenate(y_true_all_folds)
    y_pred_all = np.concatenate(y_pred_all_folds)

    cm_macro = confusion_matrix(y_true_all, y_pred_all, labels=[0, 1])  # soma/aggregado dos folds
    cm_macro_norm = cm_macro.astype(float) / cm_macro.sum(axis=1, keepdims=True)  # normalizada por linha

    # salvar CSVs macro
    base_macro = f"{nome_grupo}__{modelo_nome}__MACRO"
    caminho_cm_macro_csv = os.path.join(pasta_cm, f"{base_macro}.csv")
    caminho_cm_macro_norm_csv = os.path.join(pasta_cm, f"{base_macro}__NORMALIZADA.csv")

    pd.DataFrame(cm_macro, index=["true_0", "true_1"], columns=["pred_0", "pred_1"]).to_csv(caminho_cm_macro_csv, index=True)
    pd.DataFrame(cm_macro_norm, index=["true_0", "true_1"], columns=["pred_0", "pred_1"]).to_csv(caminho_cm_macro_norm_csv, index=True)

    # salvar figuras macro (absoluta e normalizada)
    # absoluta
    fig, ax = plt.subplots(figsize=(4, 4), dpi=120)
    ConfusionMatrixDisplay(confusion_matrix=cm_macro, display_labels=[0, 1]).plot(ax=ax, values_format='d', colorbar=False)
    ax.set_title(f"{nome_grupo} - {modelo_nome} - MACRO (absoluta)")
    fig.tight_layout()
    caminho_cm_macro_png = os.path.join(pasta_cm, f"{base_macro}.png")
    fig.savefig(caminho_cm_macro_png)
    plt.close(fig)

    # normalizada por linha
    fig, ax = plt.subplots(figsize=(4, 4), dpi=120)
    ConfusionMatrixDisplay.from_predictions(
            y_true_all, y_pred_all, display_labels=[0, 1],
            normalize='true', values_format='.2f', ax=ax, colorbar=False
        )
    ax.set_title(f"{nome_grupo} - {modelo_nome} - MACRO (normalizada)")
    fig.tight_layout()
    caminho_cm_macro_norm_png = os.path.join(pasta_cm, f"{base_macro}__NORMALIZADA.png")
    fig.savefig(caminho_cm_macro_norm_png)
    plt.close(fig)
    return caminho_cm_macro_csv,caminho_cm_macro_norm_csv,caminho_cm_macro_png,caminho_cm_macro_norm_png

In [ ]:
def cria_CM_modelo(nome_grupo, pasta_cm, modelo_nome, fold, y_test, nome_base, y_pred):
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    # Salvar CSV da matriz de confusão do experimento (fold)
    caminho_cm_csv = os.path.join(pasta_cm, f"{nome_base}.csv")
    pd.DataFrame(cm, index=["true_0", "true_1"], columns=["pred_0", "pred_1"]).to_csv(caminho_cm_csv, index=True)

    # Salvar figura da matriz de confusão do experimento
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['benigno', 'maligno'])
    fig, ax = plt.subplots(figsize=(4, 4), dpi=120)
    disp.plot(ax=ax, values_format='d', colorbar=False)
    ax.set_title(f"{nome_grupo} - {modelo_nome} - Fold {fold}")
    fig.tight_layout()
    caminho_cm_png = os.path.join(pasta_cm, f"{nome_base}.png")
    fig.savefig(caminho_cm_png)
    plt.close(fig)
    return caminho_cm_csv,caminho_cm_png

# Define modelo

In [ ]:
# Define modelo MLP
def definir_mlp_keras(input_dim):
    if input_dim >= 22000:
        units = [11000, 6000, 3000, 1500, 512]
    elif input_dim >= 10000:
        units = [6000, 3000, 1500, 512, 256]
    elif input_dim >= 5000:
        units = [3000, 1500, 512, 256, 128]
    elif input_dim > 400:
        units = [200, 100, 50]
    else:
        units = [50, 25]

    model = Sequential()
    model.add(Dense(units[0], input_shape=(input_dim,)))
    model.add(LeakyReLU(alpha=0.01))
    for u in units[1:]:
        model.add(Dense(u))
        model.add(LeakyReLU(alpha=0.01))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
# Define Modelo RF
def definir_modelos_rf_sklearn(input_dim):
    if input_dim > 20000:
        rf = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42)
    elif input_dim > 400:
        rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)
    else:
        rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)

    return rf

In [ ]:
# Define Modelo XGBoost
def definir_modelos_xgboost_sklearn(input_dim):
    if input_dim > 20000:
        xgb = XGBClassifier(n_estimators=200, max_depth=10, learning_rate=0.05, verbosity=1, use_label_encoder=False, random_state=42)
    elif input_dim > 400:
        xgb = XGBClassifier(n_estimators=150, max_depth=8, learning_rate=0.07, verbosity=1, use_label_encoder=False, random_state=42)
    else:
        xgb = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, verbosity=1, use_label_encoder=False, random_state=42)

    return xgb

# Treina Modelo

In [ ]:
# Treina, salva e faz predição com modelo MLP
def treina_modelo_mlp(pasta_saida, modelo, X_train, X_test, y_train, nome_base, callbacks):
    tempo_treino = time.perf_counter()
    history = modelo.fit(X_train, y_train, epochs=30, batch_size=256, verbose=1, validation_split=0.1, callbacks=callbacks)
    tempo_treino = time.perf_counter() - tempo_treino

    modelo.save(os.path.join(pasta_saida,nome_base+'.keras'))

    tempo_predicao = time.perf_counter()
    y_pred = modelo.predict(X_test).flatten()
    tempo_predicao = time.perf_counter() - tempo_predicao

    return y_pred, tempo_treino, tempo_predicao

In [ ]:
# Treina, salva e faz predição com modelos RF ou XGBoost
def treina_modelo_xgb_rf(pasta_saida, modelo, X_train, X_test, y_train, nome_base):
    tempo_treino = time.perf_counter()
    modelo.fit(X_train, y_train)
    tempo_treino = time.perf_counter() - tempo_treino

    joblib.dump(modelo, os.path.join(pasta_saida,nome_base+'.joblib'))

    tempo_predicao = time.perf_counter()
    y_pred = modelo.predict(X_test).astype(int)
    tempo_predicao = time.perf_counter() - tempo_predicao

    return y_pred, tempo_treino, tempo_predicao

# Avaliação de Modelo

In [ ]:
def avaliar_modelos_em_dataframe(df, modelo_nome, nome_grupo, threshold, pasta_saida, n_splits=5):
    # Dados a ser fornecido para cada modelo
    X = df.drop(columns=['classe']).values.astype(np.float32)
    y = df['classe'].astype(np.float32).values
    input_dim = X.shape[1]

    print(f"\nIniciando avaliação para o grupo de features: '{nome_grupo}' com {X.shape[1]} atributos e {X.shape[0]} instâncias.")

    # Pasta para matrizes de confusão
    pasta_cm = os.path.join(pasta_saida, "matrizes_confusao")
    os.makedirs(pasta_cm, exist_ok=True)

    # Cria a estrátegia de separação dos dados em n-folds
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)


    # Variável (list) para manter os resultados de cada fold
    resultados = []

    # Variáveis para acumular os dados para as matrizes de confusão para o modelo em execução
    y_true_all_folds = []
    y_pred_all_folds = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):

        print(f"Preparando Fold {fold}")
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        nome_base = f"{nome_grupo}__{modelo_nome}__fold{fold}"
        # Verifica o modelo
        if modelo_nome == "mlp":
          # MLP Keras
          print(f"\nIniciando MLP (Keras) para o grupo: {nome_grupo} com arquitetura dinâmica.")
          # Caso o modelo seja MLP
          print(f"\nDefinindo o modelo MLP Keras.")
          model = definir_mlp_keras(input_dim)
          early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

          with tf.device('/GPU:0'):
            ### Adicionar aqui marcação inicial de tempo de treino
            t_ini_treino = time.perf_counter()
            history = model.fit(X_train, y_train, epochs=30, batch_size=256, verbose=1, validation_split=0.1, callbacks=[early_stopping])
            ### Adicionar aqui marcação final de tempo para saber a duração do treino
            t_fim_treino = time.perf_counter()
            duracao_treino = t_fim_treino - t_ini_treino
            model.save(os.path.join(pasta_saida,nome_base+'.keras'))
            epocas_efetivas = len(history.history['loss'])
            del history
            gc.collect()

          with tf.device('/GPU:0'):
            gc.collect()


          ### Adicionar aqui marcação inicial de tempo de classificação ####
          t_ini_pred = time.perf_counter()
          y_pred_prob = model.predict(X_test).flatten()
          ### Adicionar aqui marcação final de tempo de classificação ####
          t_fim_pred = time.perf_counter()
          duracao_pred = t_fim_pred - t_ini_pred
          del model
          gc.collect()

          t = threshold

          y_pred = (y_pred_prob >= t).astype(int)
        else:
          # Modelo é RF ou XGBoost
          if modelo_nome == "rf":
              # RF
              print(f"\nIniciando RF para o grupo: {nome_grupo} com arquitetura dinâmica.")
              model = definir_modelos_rf_sklearn(input_dim)
          elif modelo_nome == "xgb":
              # XGB
              print(f"\nIniciando XGBoost para o grupo: {nome_grupo} com arquitetura dinâmica.")
              model = definir_modelos_xgboost_sklearn(input_dim)

          y_pred, duracao_treino, duracao_pred = treina_modelo_xgb_rf(pasta_saida, model, X_train, X_test, y_train, nome_base)
          del model
          gc.collect()


        # acumula para a matriz macro
        y_true_all_folds.append(y_test)
        y_pred_all_folds.append(y_pred)

        acc = accuracy_score(y_test, y_pred)
        report = classification_report(y_test.astype(int), y_pred.astype(int), output_dict=True, zero_division=0)
        print(report)

        # ===== MATRIZ DE CONFUSÃO (2x2 com labels fixos) =====
        caminho_cm_csv, caminho_cm_png = cria_CM_modelo(nome_grupo, pasta_cm, modelo_nome, fold, y_test, nome_base, y_pred)

        for classe in ['0', '1']:
            resultados.append({
                'grupo_de_features': nome_grupo,
                'modelo': f"{modelo_nome}",
                'classe': classe,
                'precision': report[classe]['precision'],
                'recall': report[classe]['recall'],
                'f1_score': report[classe]['f1-score'],
                'support': report[classe]['support'],
                'fold': fold,
                'accuracy_geral': acc,
                'tempo_treino_s': duracao_treino,
                'tempo_predicao_s': duracao_pred,
                # opcional: caminho dos artefatos do fold
                'cm_csv': caminho_cm_csv if classe == '0' else '',
                'cm_png': caminho_cm_png if classe == '0' else ''
            })
        if modelo_nome == "mlp":
          print(f"→ Threshold {t:.1f} | Fold {fold} | Acc: {acc:.4f}")
    caminho_cm_macro_csv, caminho_cm_macro_norm_csv, \
    caminho_cm_macro_png, caminho_cm_macro_norm_png = cria_CM_Macro(
        nome_grupo, pasta_cm, modelo_nome,
        y_true_all_folds, y_pred_all_folds
    )
    print(f"\nAvaliação concluída para o grupo: {nome_grupo}")
    return pd.DataFrame(resultados)


# Abertura do Arquivo

In [ ]:
from google.colab import drive
# Acesso ao drive pessoal
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Caminho para o arquivo compactado
CAMINHO_ARQUIVO = 'DIRETORIO_BASE/dados/mh1m_balanceadas_shap.npz'

# Carrega os dados com mmap_mode para uso mais leve de memória
dados = np.load(CAMINHO_ARQUIVO, allow_pickle=True)

# Extração dos arrays principais
X = dados['data']
y = dados['classes']
colunas = dados['column_names']

# Embaralhar X e y
rng = np.random.default_rng(42)  # garante reprodutibilidade
idx_final = rng.permutation(X.shape[0])  # embaralha os índices

X = X[idx_final]
y = y[idx_final]

print(f"Dados embaralhados: X={X.shape}, y={y.shape}")

Dados embaralhados: X=(159020, 4284), y=(159020,)


# Definição do grupos e modelos

In [ ]:
modelos_nome = ["mlp", "rf", "xgb"]
threshold = 0.5
grupos = ["intents", "permissions", "opcodes", "apicalls", "permissions_opcodes", "todas"]


# Executar

In [ ]:
CAMINHO_RAIZ = "DIRETORIO_BASE/src_shap/modelos"

for modelo_nome in modelos_nome:
  for nome_grupo in grupos:
      if modelo_nome == "mlp":
        pasta_saida = os.path.join(CAMINHO_RAIZ,nome_grupo,f"{modelo_nome}_limiar{trunc(threshold*10)}","resultados")
      else:
        pasta_saida = os.path.join(CAMINHO_RAIZ,nome_grupo,f"{modelo_nome}","resultados")

      os.makedirs(pasta_saida, exist_ok=True)

      # Parte 3 - Separar as colunas das features e criar os DataFrames
      # Identificar colunas por namespace ("intents, permissions, opcodes, apicalls")
      if nome_grupo == "permissions_opcodes":
          idx_permissions = [i for i, nome in enumerate(colunas) if nome.startswith("permissions::")]
          idx_opcodes = [i for i, nome in enumerate(colunas) if nome.startswith("opcodes::")]
          idx_features = idx_permissions + idx_opcodes
          df = pd.DataFrame(X[:, idx_features], columns=np.array(colunas)[idx_features])
          df['classe'] = y
      elif nome_grupo == "todas":
          idx_features = range(len(colunas))
          df = pd.DataFrame(X, columns=np.array(colunas)[idx_features])
          df['classe'] = y
      else:
          idx_features = [i for i, nome in enumerate(colunas) if nome.startswith(f"{nome_grupo}::")]
          df = pd.DataFrame(X[:, idx_features], columns=np.array(colunas)[idx_features])
          df['classe'] = y


      print("DataFrames criados:")
      print(f" - df : {df.shape}")

      # Parte 7 - Executar o modelo e recuperar os resultados para cada DataFrame
      df_resultados = pd.concat([
          avaliar_modelos_em_dataframe(df, modelo_nome, nome_grupo, threshold, pasta_saida, 5),
      ], ignore_index=True)


      # Parte 8.2 - Exportar os dados consolidados
      caminho_saida = os.path.join(pasta_saida, 'resultados_modelos.csv')
      df_resultados.to_csv(caminho_saida, index=False)

      resumo = df_resultados.groupby(['grupo_de_features', 'modelo', 'classe'])[['precision', 'recall', 'f1_score']].mean().round(4)
      resumo.to_csv(os.path.join(pasta_saida, 'resumo_resultados.csv'))

      # Parte 8.3 - Exibir e salvar resumo
      print(resumo)

      resumo.to_csv(os.path.join(pasta_saida, 'resumo_resultados.csv'))

      print(f"\nArquivos salvos em: {pasta_saida}")
      print("• CSV por experimento da matriz de confusão em: pasta 'matrizes_confusao/'")
      print("• PNG por experimento da matriz de confusão em: pasta 'matrizes_confusao/'")